In [15]:
# E) Graded-10 with train_focus2/best
import json
import os
from datetime import UTC, datetime
from pathlib import Path

import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

from src.trainer import GRADED_QUESTIONS

os.chdir("/content/USU-LLM-Class-F26")
MODEL_ID = "meta-llama/Llama-3.2-1B-Instruct"
ADAPTER = Path("data/assignment6/train_focus2/best")
OUT = Path("data/assignment6/after_focus2_train.json")
DEVICE = "cuda"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, attn_implementation="sdpa"
)
model = PeftModel.from_pretrained(base, ADAPTER)
model.to(DEVICE)
model.eval()


def ask(question: str) -> str:
    messages = [{"role": "user", "content": question}]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    with torch.inference_mode():
        out = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    new_tokens = out[0, inputs["input_ids"].shape[-1] :]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


examples = []
for q in GRADED_QUESTIONS:
    ans = ask(q)
    examples.append({"question": q, "answer": ans})
    print("Q:", q)
    print("A:", ans)
    print("---")

payload = {
    "model": MODEL_ID,
    "adapter": str(ADAPTER),
    "device": DEVICE,
    "max_new_tokens": 128,
    "generation": "greedy",
    "created_utc": datetime.now(UTC).isoformat(),
    "examples": examples,
}
OUT.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n")
print("Wrote", OUT)

Q: If it takes 1 hour for 60 people to play an Opera, how many hours will it take 600 people to play the same opera?
A: It still takes 1 hour for 60 people to play the opera; the number of people does not change the time it takes to play the opera.
---
Q: Is a pound of feathers or a British pound heavier?
A: A pound is a unit of mass, not weight. The British pound is a currency unit.
---
Q: A boy runs down the stairs in the morning and sees a tree in his living room, and some boxes under the tree. What's going on?
A: The boy is playing a game of hide-and-seek.
---
Q: What happens if you crack your knuckles a lot?
A: It can cause joint pain and swelling.
---
Q: If there is a shark in the pool of my basement, is it safe to go upstairs?
A: Yes, it is safe to go upstairs.
---
Q: How much wood could a wood chuck chuck if there were only 5 pounds of wood in the world?
A: About 5 pounds. The question is a riddle, not a problem.
---
Q: Who is the current President of the United States?
A: Dona